In [1]:
import pickle
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

In [23]:
# ============================================================
# RDKit descriptor calculator
# ============================================================

descriptor_names = [
    name for name, _
    in Descriptors._descList
]

calculator = MoleculeDescriptors.MolecularDescriptorCalculator(
    descriptor_names
)


# ============================================================
# SMILES preprocessing
# ============================================================

def canonicalize_smiles(smiles, keep_largest=True):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    if keep_largest and "." in smiles:

        fragments = Chem.GetMolFrags(
            mol,
            asMols=True
        )

        mol = max(
            fragments,
            key=lambda m: m.GetNumHeavyAtoms()
        )

    return Chem.MolToSmiles(
        mol,
        canonical=True
    )


# ============================================================
# Descriptor calculation
# ============================================================

def compute_rdkit_descriptors(smiles):

    mol = Chem.MolFromSmiles(smiles)

    values = calculator.CalcDescriptors(mol)

    return {
        f"rdkit-{name}": value
        for name, value in zip(
            descriptor_names,
            values
        )
    }


# ============================================================
# Predictor class
# ============================================================

class SurfProPredictor:

    def __init__(self, model_file, target_labels=None):

        with open(model_file, "rb") as f:
            self.results = pickle.load(f)

        # 🔑 MUST define this first
        self.targets = list(self.results.keys())

        # 🎯 default labels
        default_labels = {
            "pCMC": "pCMC",
            "AW_ST_CMC": r"$\gamma_{\mathrm{CMC}}$",
            "Gamma_max": r"$\Gamma_{\max}$",
            "pC20": "pC$_{20}$",
            "Pi_CMC": r"$\Pi_{\mathrm{CMC}}$",
            "Area_min": r"A$_{\min}$",
            "viscosity": r"$\eta$",
            "D_MOL": r"D$_{\mathrm{MOL}}$",
            "D_SOL": r"D$_{\mathrm{SOL}}$",
            "surface_tension_avg": r"$\gamma$",
        }

        # merge user + default safely
        if target_labels is None:
            self.target_labels = default_labels
        else:
            self.target_labels = target_labels
    
    def _build_features(
        self,
        smiles,
        feature_schema
    ):

        smiles = canonicalize_smiles(smiles)
        descriptors = compute_rdkit_descriptors(
            smiles
        )

        X = pd.DataFrame([descriptors])
        X = X.reindex(
            columns=feature_schema
        )
        return X

    def predict_target(
        self,
        smiles,
        target,
        test_id=0
    ):

        info = self.results[target][test_id]
        feature_schema = info["feature_schema"]

        X = self._build_features(
            smiles,
            feature_schema
        )

        fold_predictions = []

        for fold in info["fold_models"]:

            scaler = fold["scaler"]
            model = fold["model"]

            X_scaled = scaler.transform(X)

            pred = model.inplace_predict(X_scaled)

            fold_predictions.append(pred[0])

        return np.mean(fold_predictions)

    def predict(
        self,
        smiles,
        test_id=0,
        return_ci=False,
        return_explanation=False,
        shap_max_folds=3
        ):
    
        # =========================
        # 1. BASE PREDICTIONS
        # =========================
        predictions = {}
    
        for target in self.targets:
    
            predictions[target] = self.predict_target(
                smiles,
                target,
                test_id=test_id
            )
    
        output = {
            "prediction": pd.Series(
                self._format_results(predictions)
            )
        }
    
        # =========================
        # 2. UNCERTAINTY (CI)
        # =========================
        if return_ci:
    
            ci = self.predict_with_error_bars(smiles)
    
            # apply labels
            ci = ci.copy()
            ci.index = [
                self.target_labels.get(i, i)
                for i in ci.index
            ]
    
            output["uncertainty"] = ci
    
        # =========================
        # 3. EXPLANATION (SHAP)
        # =========================
        if return_explanation:
    
            explanations = {}
    
            for target in self.targets:
    
                mean, std = self.explain_target(
                    smiles,
                    target,
                    test_id=test_id,
                    max_folds=shap_max_folds,
                    return_std=True
                )
    
                label = self.target_labels.get(target, target)
    
                explanations[label] = {
                    "top_features": mean.head(5),
                    "top_feature_uncertainty": std.loc[mean.head(5).index]
                }
    
            output["explanation"] = explanations
    
        return output
        
    def _format_results(self, results_dict):
    
        formatted = {}
    
        for key, value in results_dict.items():
    
            label = self.target_labels.get(key, key)
    
            formatted[label] = value
    
        return formatted
    
    def predict_with_error_bars(self, smiles):
    
        per_target = {}
    
        for target in self.targets:
    
            split_means = []
    
            for test_id in self.results[target]:
    
                info = self.results[target][test_id]
    
                feature_schema = info["feature_schema"]
    
                X = self._build_features(smiles, feature_schema)
    
                fold_preds = []
    
                for fold in info["fold_models"]:
    
                    scaler = fold["scaler"]
                    model = fold["model"]
    
                    Xs = scaler.transform(X)
                    pred = model.inplace_predict(Xs)[0]
    
                    fold_preds.append(pred)
    
                split_means.append(np.mean(fold_preds))
    
            split_means = np.array(split_means)
    
            per_target[target] = {
                "mean": float(np.mean(split_means)),
                "std": float(np.std(split_means)),
                "min": float(np.min(split_means)),
                "max": float(np.max(split_means)),
            }
    
        # 👇 apply pretty labels
        formatted = {}
    
        for k, v in per_target.items():
            label = self.target_labels.get(k, k)
            formatted[label] = v
    
        return pd.DataFrame(formatted).T

    def explain_target(
        self,
        smiles,
        target,
        test_id=0,
        fold_ids=None,
        max_folds=3,
        return_std=True
        ):

        import shap
    
        info = self.results[target][test_id]
        feature_schema = info["feature_schema"]
    
        X = self._build_features(smiles, feature_schema)
    
        fold_ids = fold_ids or list(range(len(info["fold_models"])))
        fold_ids = fold_ids[:max_folds]  # 👈 speed control
    
        shap_values_all = []
    
        for fold_id in fold_ids:
    
            fold = info["fold_models"][fold_id]
    
            scaler = fold["scaler"]
            model = fold["model"]
    
            Xs = scaler.transform(X)
    
            explainer = shap.TreeExplainer(model.get_booster())
            shap_values = explainer.shap_values(Xs)[0]
    
            shap_values_all.append(np.abs(shap_values))
    
        shap_values_all = np.array(shap_values_all)
    
        mean_shap = shap_values_all.mean(axis=0)
    
        result = pd.Series(
            mean_shap,
            index=feature_schema
        ).sort_values(ascending=False)
    
        if return_std:
            std_shap = shap_values_all.std(axis=0)
    
            std_series = pd.Series(
                std_shap,
                index=feature_schema
            ).loc[result.index]
    
            return result, std_series
    
        return result

        def explain(
            self,
            smiles,
            test_id=0,
            max_folds=3
        ):
        
            results = {}
        
            for target in self.targets:
        
                mean, std = self.explain_target(
                    smiles,
                    target,
                    test_id=test_id,
                    max_folds=max_folds,
                    return_std=True
                )
        
                results[target] = {
                    "mean_importance": mean.head(5),
                    "std_importance": std.head(5)
                }
        
            return results

        def print_explanation(self, smiles, max_folds=3):
        
            results = self.explain(smiles, max_folds=max_folds)
        
            for target, res in results.items():
        
                label = self.target_labels.get(target, target)
        
                print("\n", "="*60)
                print(label)
                print("="*60)
        
                for feat in res["mean_importance"].index:
        
                    mean = res["mean_importance"][feat]
                    std = res["std_importance"][feat]
        
                    print(f"{feat:30s} {mean:.4f} ± {std:.4f}")

In [24]:
predictor = SurfProPredictor("models.pkl")

smiles = "CCCCCCCC(=O)O"

prediction = predictor.predict_with_error_bars("CCCCCCCC(=O)[OH]")

print(prediction)

predictor.predict(smiles,
                   return_ci=True,
                   return_explanation=True)

                               mean       std         min         max
pCMC                       0.904706  0.487147    0.454760    1.830358
$\gamma_{\mathrm{CMC}}$   31.356176  0.589198   30.855642   32.394478
$\Gamma_{\max}$            3.555962  0.207476    3.191123    3.752637
A$_{\min}$                 0.959463  0.225173    0.705206    1.311720
pC$_{20}$                  3.188890  0.318323    2.674531    3.584527
D$_{\mathrm{MOL}}$         0.507350  0.018233    0.484829    0.537030
D$_{\mathrm{SOL}}$         3.119607  0.027821    3.064449    3.137911
$\gamma$                 493.138855  1.578113  491.320221  495.208496
$\eta$                     0.000781  0.000002    0.000777    0.000783


ValueError: could not convert string to float: '[3.2084293E0]'

In [11]:
with open("models.pkl", "rb") as f:
    models = pickle.load(f)

print(models.keys())

dict_keys(['pCMC', 'AW_ST_CMC', 'Gamma_max', 'Area_min', 'pC20', 'D_MOL', 'D_SOL', 'surface_tension_avg', 'viscosity'])
